[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/S11_01_cnn.ipynb)

# S11 · Práctica 1 — Redes convolucionales (CNN): cómo "ve" una red

**Tiempo estimado:** 1 h 45 min · **Sprint 11** (UD 4.4)

**Contexto TurisData.** El cliente tiene miles de fotos de alojamientos, zonas comunes y platos, y hoy
nadie las usa. Antes de clasificarlas necesitas entender cómo una red **procesa una imagen**. Aquí lo verás
con imágenes pequeñas que se ejecutan sin internet.

**Al terminar sabrás:**
1. Qué es una imagen para un ordenador (matriz de números) y qué hace una **convolución** y un **pooling**.
2. Calcular a mano el tamaño de salida de una capa convolucional.
3. Construir, entrenar y evaluar una CNN pequeña con Keras y compararla con un MLP.
4. Mirar dentro de la red: filtros y mapas de activación; y usar *data augmentation*.

**Dos ramas.** (a) *Rama offline (todo el notebook)*: imágenes de dígitos 8×8 incluidas en scikit-learn.
(b) *Rama Colab (una celda marcada "requiere internet")*: descarga de `tf_flowers` para probar con fotos reales.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score

SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)

## 1. Una imagen es una matriz de números

**Idea.** Una imagen en escala de grises es una tabla de píxeles con valores (aquí de 0 = negro a 16 = blanco).
Una imagen en color tiene **3 canales** (rojo, verde, azul): una tabla por color. Su forma es
`(alto, ancho, canales)`. Un lote de imágenes añade una dimensión delante: `(n_imágenes, alto, ancho, canales)`.

In [ ]:
digitos = load_digits()
imagenes, etiquetas = digitos.images, digitos.target      # (1797, 8, 8) y (1797,)
print("Forma del conjunto:", imagenes.shape, "· valores de", imagenes.min(), "a", imagenes.max())
print("Un dígito (matriz 8×8):\n", imagenes[0].astype(int))

fig, ejes = plt.subplots(2, 8, figsize=(12, 3.4))
for i, eje in enumerate(ejes.ravel()):
    eje.imshow(imagenes[i], cmap="gray_r"); eje.set_title(f"es un {etiquetas[i]}", fontsize=9); eje.axis("off")
plt.suptitle("Dígitos 8×8 de scikit-learn (miniaturas de 64 píxeles)"); plt.tight_layout(); plt.show()

**Ejercicio 1.** Prepara los datos para una red: en `X` guarda las imágenes **normalizadas entre 0 y 1**
(divide entre 16) con forma `(n, 8, 8, 1)` (Keras espera un canal explícito). Después crea con
`train_test_split` (`test_size=0.25`, `random_state=SEMILLA`, `stratify=etiquetas`) los conjuntos
`X_ent, X_pru, y_ent, y_pru`.

In [ ]:
# TODO: imagenes / 16.0 y añade el eje del canal con [..., np.newaxis]
...

assert X.shape == (1797, 8, 8, 1), "X debe tener forma (1797, 8, 8, 1): ¿has añadido el canal?"
assert X.max() <= 1.0 and X.min() >= 0.0, "Normaliza dividiendo entre 16 para que quede entre 0 y 1"
assert len(X_ent) + len(X_pru) == 1797 and len(X_pru) == 450, "Reparto 75 % / 25 %"

## 2. La convolución: un filtro que se desliza

**Idea.** Un **filtro** (o *kernel*) es una mini-tabla, por ejemplo 3×3. Se coloca sobre una zona de la imagen,
se multiplica casilla a casilla, se suma y se obtiene **un** número. Después se desliza una posición y se repite.
El resultado es un **mapa de activación**: dice *dónde* aparece el patrón que busca el filtro (un borde vertical,
una esquina…). Una CNN **aprende sus propios filtros** durante el entrenamiento.

Vamos a hacerlo a mano con NumPy sobre una imagen sencilla: un rectángulo claro en fondo oscuro.

In [ ]:
def convolucion2d(imagen, filtro):
    """Convolución 'válida' (sin relleno, paso 1) de una imagen 2D con un filtro 2D."""
    alto, ancho = imagen.shape
    k_alto, k_ancho = filtro.shape
    salida = np.zeros((alto - k_alto + 1, ancho - k_ancho + 1))
    for i in range(salida.shape[0]):
        for j in range(salida.shape[1]):
            zona = imagen[i:i + k_alto, j:j + k_ancho]
            salida[i, j] = np.sum(zona * filtro)
    return salida

**Ejercicio 2 (calcular el tamaño de salida).** Para una entrada de `n` píxeles, filtro de `k`, relleno (*padding*) `p`
a cada lado y paso (*stride*) `s`, la salida mide `(n - k + 2p) / s + 1` (parte entera). Escribe
`tam_salida(n, k, p=0, s=1)` que devuelva ese número entero. Sirve para entender por qué las imágenes "encogen"
capa tras capa (y por qué Keras ofrece `padding="same"`).

In [ ]:
# TODO: Aplica la fórmula con división entera (//)
...

assert tam_salida(8, 3) == 6, "8 píxeles con filtro 3 sin relleno dan 6"
assert tam_salida(8, 3, p=1) == 8, "Con relleno 1 el tamaño se conserva ('same')"
assert tam_salida(224, 7, p=3, s=2) == 112, "Caso típico de las primeras capas de ResNet: 224 → 112"

In [ ]:
# Imagen de prueba: rectángulo claro sobre fondo oscuro
prueba = np.zeros((14, 14)); prueba[3:11, 4:10] = 1.0
filtro_vertical = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])     # detecta bordes verticales (Sobel)
filtro_horizontal = filtro_vertical.T                                  # detecta bordes horizontales

fig, ejes = plt.subplots(1, 3, figsize=(11, 3.6))
for eje, im, titulo in zip(ejes, [prueba, convolucion2d(prueba, filtro_vertical), convolucion2d(prueba, filtro_horizontal)],
                           ["Imagen original", "Bordes verticales", "Bordes horizontales"]):
    eje.imshow(im, cmap="gray"); eje.set_title(titulo); eje.axis("off")
plt.tight_layout(); plt.show()
print("Tamaño de la salida:", convolucion2d(prueba, filtro_vertical).shape, "(14 - 3 + 1 = 12)")

## 3. Pooling: quedarse con lo importante

**Idea.** El *max pooling* divide el mapa en bloques (p. ej. 2×2) y se queda con el **máximo** de cada uno.
Resultado: mapas la mitad de grandes, menos cálculo y algo de **tolerancia a pequeños desplazamientos**
(si el borde se mueve un píxel dentro del bloque, el máximo no cambia).

**Ejercicio 3.** Escribe `max_pooling(mapa, t=2)` que divida un mapa 2D (con lados múltiplos de `t`) en bloques de
`t × t` y devuelva el máximo de cada bloque (salida de lado `alto // t`). Pista: recorre los bloques con dos bucles
o busca cómo reorganizar el mapa con `reshape` para agrupar cada bloque.

In [ ]:
# TODO: Dos bucles sobre los bloques (np.max de cada zona) o un reshape a 4 dimensiones
...

_m = np.array([[1, 2, 0, 0], [3, 4, 0, 1], [0, 0, 5, 6], [0, 9, 7, 8]])
assert np.array_equal(max_pooling(_m), [[4, 1], [9, 8]]), "Esperado [[4, 1], [9, 8]]: máximo de cada bloque 2×2"
assert max_pooling(np.zeros((12, 12))).shape == (6, 6), "Una imagen de 12×12 con bloques 2×2 debe dar 6×6"

In [ ]:
salida_conv = convolucion2d(prueba, filtro_vertical)
fig, ejes = plt.subplots(1, 2, figsize=(8, 3.6))
ejes[0].imshow(np.abs(salida_conv), cmap="gray"); ejes[0].set_title("Mapa 12×12 tras la convolución"); ejes[0].axis("off")
ejes[1].imshow(max_pooling(np.abs(salida_conv)), cmap="gray"); ejes[1].set_title("Tras max pooling 2×2 (6×6)"); ejes[1].axis("off")
plt.tight_layout(); plt.show()

## 4. Una CNN completa en Keras

**Arquitectura típica:** `[Conv → ReLU → Pooling] × varias veces` → **Flatten** (aplanar) → `Dense` → salida.
Las primeras capas detectan trazos simples; las siguientes los combinan en formas cada vez más complejas
(es la idea de LeNet, VGG o ResNet, solo que mucho más grandes).

Como referencia entrenaremos también un **MLP** que trata la imagen como una lista de 64 números y
**pierde la vecindad entre píxeles**.

In [ ]:
def construir_mlp_imagen():
    m = keras.Sequential([keras.layers.Input(shape=(8, 8, 1)), keras.layers.Flatten(),
                          keras.layers.Dense(64, activation="relu"), keras.layers.Dense(10, activation="softmax")])
    m.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

mlp = construir_mlp_imagen()
h_mlp = mlp.fit(X_ent, y_ent, validation_split=0.2, epochs=25, batch_size=32, verbose=0)
print("MLP · parámetros:", mlp.count_params(), "· precisión en prueba:", round(mlp.evaluate(X_pru, y_pru, verbose=0)[1], 3))

**Ejercicio 4.** Escribe `construir_cnn(filtros=(16, 32))` que devuelva un modelo compilado con:
`Input(8,8,1)` → por cada valor de `filtros`: `Conv2D(f, 3, padding="same", activation="relu")` +
`MaxPooling2D(2)` → `Flatten` → `Dense(10, activation="softmax")`.
Compílalo igual que el MLP (`adam`, `sparse_categorical_crossentropy`, métrica `accuracy`).

In [ ]:
# TODO: Un bucle sobre filtros añadiendo Conv2D y MaxPooling2D; después Flatten y Dense(10, softmax)
...

cnn = construir_cnn()
cnn.summary()
assert sum(isinstance(c, keras.layers.Conv2D) for c in cnn.layers) == 2, "Deben ser 2 capas convolucionales"
assert sum(isinstance(c, keras.layers.MaxPooling2D) for c in cnn.layers) == 2, "Deben ser 2 capas de max pooling"
assert cnn.output_shape == (None, 10), "La salida debe tener 10 neuronas (una por dígito)"
assert cnn.count_params() == (3*3*1*16 + 16) + (3*3*16*32 + 32) + (2*2*32*10 + 10), "Parámetros inesperados: revisa filtros, padding='same' y el orden de las capas"

In [ ]:
keras.utils.set_random_seed(SEMILLA)
cnn = construir_cnn()
h_cnn = cnn.fit(X_ent, y_ent, validation_split=0.2, epochs=30, batch_size=32, verbose=0)
acc_cnn = cnn.evaluate(X_pru, y_pru, verbose=0)[1]
acc_mlp = mlp.evaluate(X_pru, y_pru, verbose=0)[1]

plt.plot(h_cnn.history["val_accuracy"], label="CNN"); plt.plot(h_mlp.history["val_accuracy"], label="MLP")
plt.xlabel("época"); plt.ylabel("precisión de validación"); plt.title("CNN frente a MLP en dígitos 8×8"); plt.legend(); plt.show()
print(f"MLP: {mlp.count_params()} parámetros, precisión {acc_mlp:.3f}")
print(f"CNN: {cnn.count_params()} parámetros, precisión {acc_cnn:.3f}")

**Qué observar.** En imágenes minúsculas y limpias un MLP ya rinde muy bien, así que las precisiones serán parecidas.
La ventaja de la CNN crece con imágenes más grandes, con objetos desplazados o con ruido: comparte los mismos filtros
en toda la imagen y respeta la vecindad. Fíjate en los parámetros: aquí (8×8) el MLP no es más grande que la CNN, pero con
fotos de 96×96×3 (27 648 valores de entrada) un MLP con solo 64 neuronas en la primera capa ya necesitaría ≈ 1,8 millones
de pesos, mientras que una capa convolucional de 3×3 con 16 filtros usa 448, sea cual sea el tamaño de la foto.
Comprueba la tolerancia a desplazamientos en la sección 6.

## 5. Mirar dentro de la red
Los **filtros** de la primera capa son pequeñas plantillas de trazos; los **mapas de activación** muestran dónde
se activa cada filtro con un dígito concreto.

In [ ]:
filtros_1 = cnn.layers[0].get_weights()[0]           # forma (3, 3, 1, 16)
fig, ejes = plt.subplots(1, 8, figsize=(12, 2))
for i, eje in enumerate(ejes):   # mostramos los 8 primeros filtros
    eje.imshow(filtros_1[:, :, 0, i], cmap="coolwarm"); eje.set_title(f"filtro {i}", fontsize=8); eje.axis("off")
plt.suptitle("Filtros 3×3 aprendidos por la primera capa (rojo = pesa positivo, azul = negativo)"); plt.show()

extractor = keras.Model(inputs=cnn.inputs, outputs=cnn.layers[0].output)
ejemplo = X_pru[:1]
mapas = extractor.predict(ejemplo, verbose=0)[0]     # (8, 8, 16)
fig, ejes = plt.subplots(1, 9, figsize=(13, 2))
ejes[0].imshow(ejemplo[0, :, :, 0], cmap="gray_r"); ejes[0].set_title(f"entrada ({y_pru[0]})", fontsize=8); ejes[0].axis("off")
for i in range(8):
    ejes[i + 1].imshow(mapas[:, :, i], cmap="viridis"); ejes[i + 1].set_title(f"mapa {i}", fontsize=8); ejes[i + 1].axis("off")
plt.show()

## 6. *Data augmentation*: enseñar a la red que la imagen puede moverse

**Idea.** Si el alojamiento aparece más a la izquierda, más pequeño o ligeramente girado, sigue siendo el mismo
alojamiento. El *data augmentation* genera **variaciones aleatorias** del entrenamiento (solo al entrenar) para que
la red no dependa de la posición exacta. Keras lo hace con capas (`RandomTranslation`, `RandomRotation`, `RandomZoom`,
`RandomFlip`) que se colocan al principio del modelo.

Prueba de estrés: evaluaremos ambos modelos con **dígitos desplazados un píxel** (una situación realista que
el entrenamiento no vio).

In [ ]:
def desplazar(imagenes_, dx, dy):
    return np.roll(np.roll(imagenes_, dy, axis=1), dx, axis=2)

X_pru_desp = desplazar(X_pru, 1, 1)

aumento = keras.Sequential([keras.layers.RandomTranslation(0.12, 0.12, fill_mode="constant"),
                            keras.layers.RandomRotation(0.06, fill_mode="constant")])
lote = X_ent[:10]
fig, ejes = plt.subplots(2, 10, figsize=(13, 3))
for j in range(10):
    ejes[0, j].imshow(lote[j, :, :, 0], cmap="gray_r"); ejes[0, j].axis("off")
    ejes[1, j].imshow(aumento(lote[j:j + 1], training=True)[0, :, :, 0], cmap="gray_r"); ejes[1, j].axis("off")
ejes[0, 0].set_title("original", fontsize=8, loc="left"); ejes[1, 0].set_title("aumentada", fontsize=8, loc="left")
plt.show()

**Mini-reto integrador.** Construye `cnn_aug`: una CNN igual que `construir_cnn()` pero con las capas de `aumento`
**al principio** del modelo (usa `keras.Sequential([keras.layers.Input((8, 8, 1)), aumento, ...])` o añádelas con `add`).
Entrénala 40 épocas (`epochs=40`, `batch_size=32`, `validation_split=0.2`) y guarda en `acc_aug_desp` su precisión
sobre `X_pru_desp`. Guarda también en `acc_cnn_desp` la de la CNN sin aumento. Compáralas en la celda de texto.

In [ ]:
# TODO: Sequential con Input, las capas de aumento, y después las Conv/Pool/Flatten/Dense; compila igual
...

assert any(isinstance(c, keras.Sequential) or "random" in c.name for c in cnn_aug.layers), "Falta incluir las capas de aumento dentro del modelo"
assert 0.0 <= acc_aug_desp <= 1.0 and 0.0 <= acc_cnn_desp <= 1.0, "Las precisiones deben estar entre 0 y 1"
print(f"Con dígitos desplazados → CNN normal: {acc_cnn_desp:.3f} · CNN con aumento: {acc_aug_desp:.3f} · MLP: {mlp.evaluate(X_pru_desp, y_pru, verbose=0)[1]:.3f}")

**Tu conclusión (edita esta celda):** …

### Matriz de confusión de la CNN (dónde se equivoca)

In [ ]:
pred = cnn.predict(X_pru, verbose=0).argmax(axis=1)
cm = confusion_matrix(y_pru, pred)
plt.figure(figsize=(5.5, 5)); plt.imshow(cm, cmap="Blues")
for i in range(10):
    for j in range(10):
        if cm[i, j]:
            plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xticks(range(10)); plt.yticks(range(10)); plt.xlabel("predicho"); plt.ylabel("real"); plt.title("Matriz de confusión (CNN)"); plt.show()

## 7. Rama Colab (opcional): fotos reales con `tf_flowers`

> **AVISO: requiere internet. No se ejecuta en el taller sin conexión.** Está pensada para Colab en casa.
> Descarga ≈ 220 MB de fotos de 5 tipos de flores. Cambia `USAR_INTERNET` a `True` para probarla.
> **Esta celda no se ha verificado en el entorno del taller (sin internet).**

In [ ]:
USAR_INTERNET = False        # ← cámbialo a True solo si tienes conexión (Colab)

if USAR_INTERNET:
    ruta_flores = keras.utils.get_file(
        "flower_photos", untar=True,
        origin="https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz")
    ruta_flores = os.path.join(os.path.dirname(ruta_flores), "flower_photos")
    ds_ent, ds_val = keras.utils.image_dataset_from_directory(
        ruta_flores, validation_split=0.2, subset="both", seed=SEMILLA, image_size=(96, 96), batch_size=32)
    print("Clases:", ds_ent.class_names)
    cnn_flores = keras.Sequential([
        keras.layers.Input((96, 96, 3)), keras.layers.Rescaling(1 / 255),
        keras.layers.RandomFlip("horizontal"),
        keras.layers.Conv2D(16, 3, activation="relu"), keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(32, 3, activation="relu"), keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(64, 3, activation="relu"), keras.layers.MaxPooling2D(),
        keras.layers.Flatten(), keras.layers.Dropout(0.3), keras.layers.Dense(5, activation="softmax")])
    cnn_flores.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
    cnn_flores.fit(ds_ent, validation_data=ds_val, epochs=8)
else:
    print("Rama de Colab desactivada (USAR_INTERNET = False). Sigue con el resto del notebook.")

## 8. Preguntas de reflexión
1. ¿Por qué, en imágenes grandes, una CNN necesita muchos menos parámetros que un MLP?
2. Si aplicamos un filtro 5×5 sin relleno a una imagen de 32×32, ¿qué tamaño tiene la salida? ¿Y con `padding="same"`?
3. ¿En qué casos sería un error aplicar `RandomFlip("horizontal")` a las fotos de TurisData? (Piensa en carteles con texto.)
4. ¿Por qué el *data augmentation* no se aplica al conjunto de prueba?

## 9. Resumen: qué has aprendido
- Una imagen es un tensor `(alto, ancho, canales)`; se normaliza antes de entrenar.
- Convolución = filtro deslizante que detecta patrones; pooling = reducir quedándose con lo más fuerte.
- Arquitectura típica: `[Conv → ReLU → Pool] × n → Flatten → Dense`.
- Una CNN comparte filtros por toda la imagen: menos parámetros y mejor tolerancia a desplazamientos.
- *Data augmentation* = variaciones aleatorias solo en entrenamiento para generalizar mejor.